# MeetMe: Personality Fine-Tuning with Qwen2.5-3B and Unsloth

This notebook provides a full pipeline to fine-tune `Qwen2.5-3B-Instruct` to replicate a specific personality using QLoRA. We utilize **Unsloth** for 2x faster training and 70% less memory usage, optimized for Google Colab's T4 GPU.

## 1. Install Dependencies
We install `unsloth`, `xformers`, and the Hugging Face ecosystem (`trl`, `peft`, `accelerate`, `transformers`).

In [5]:
%%capture
# Forced re-installation to ensure consistency in Colab
!pip install --upgrade "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes

# Standard Colab fix: restart session if imports fail after this cell
import os
# os.kill(os.getpid(), 9)

## 2. Load Model & Tokenizer
We load the model in **4-bit quantization** to fit within the 16GB VRAM limit of the T4 GPU.

In [7]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None # None for auto detection
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen2.5-3B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 32, # Higher rank for personality nuance
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0, # Optimized for Unsloth
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

print("Model loaded with QLoRA adapters.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.7.6: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Unsloth 2026.7.6 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


Model loaded with QLoRA adapters.


## 3. Data Loading & Validation
We load `train_dataset.json` and perform quality checks to ensure valid JSON structure and non-empty messages.

In [8]:
import json
import os
from datasets import Dataset

dataset_path = 'train_dataset.json'

if not os.path.exists(dataset_path):
    print(f"ERROR: {dataset_path} not found. Please upload it to the /content/ directory.")
else:
    with open(dataset_path, 'r') as f:
        raw_data = json.load(f)

    valid_data = []
    for i, entry in enumerate(raw_data):
        messages = entry.get("messages", [])
        if not messages or len(messages) < 2:
            continue
        if any(not m.get("content", "").strip() for m in messages):
            continue
        valid_data.append(entry)

    print(f"Passed validation: {len(valid_data)}/{len(raw_data)} samples.")
    dataset = Dataset.from_list(valid_data)

    def formatting_prompts_func(examples):
        convs = examples["messages"]
        texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convs]
        return { "text" : texts, }

    dataset = dataset.map(formatting_prompts_func, batched = True)

Passed validation: 171/171 samples.


Map:   0%|          | 0/171 [00:00<?, ? examples/s]

## 4. Training Configuration
We use a learning rate of 2e-4 and 3-5 epochs. `Gradient Accumulation` helps simulate a larger batch size on the T4.

In [9]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # Can be True for faster training
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = -1,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        save_strategy = "no", # Change to 'epoch' for longer runs
    ),
)

trainer_stats = trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/171 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 171 | Num Epochs = 3 | Total steps = 66
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 59,867,136 of 3,145,805,824 (1.90% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
1,4.797421
2,4.545389
3,4.204610
4,3.733063
5,2.474288
6,2.249625
7,2.260145
8,2.453040
9,2.499377
10,2.038056


## 5. Inference & Evaluation
Now that training is complete, we can chat with the model to evaluate its personality and reasoning. We use `FastLanguageModel.for_inference` to optimize for speed.

In [12]:
from transformers import TextStreamer

# Prepare for inference
FastLanguageModel.for_inference(model)

messages = [
    {"role": "user", "content": "What is your philosophy on personal growth and learning?"},
]

# Explicitly generating attention_mask to avoid warnings
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt",
).to("cuda")

# Adding attention_mask explicitly
text_streamer = TextStreamer(tokenizer)
_ = model.generate(
    input_ids = inputs,
    attention_mask = (inputs != tokenizer.pad_token_id).long(),
    streamer = text_streamer,
    max_new_tokens = 512,
    temperature = 0.7,
    use_cache = True
)

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
What is your philosophy on personal growth and learning?<|im_end|>
<|im_start|>assistant


Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


My philosophy is that continuous learning is more important than finishing something quickly. I want to understand how things work rather than just memorizing them. I enjoy building systems that can explain complex ideas clearly because it shows me what I still don't understand.<|im_end|>


## 6. Save the Model
You can save the LoRA adapters to disk or upload them to Hugging Face.

In [13]:
# 1. Save LoRA adapters (lightweight)
model.save_pretrained("meetme_lora_model")
tokenizer.save_pretrained("meetme_lora_model")

# 2. Merge to 16bit for production/deployment (Optional but recommended)
# This merges the LoRA weights into the base model
# model.save_pretrained_merged("meetme_merged_model", tokenizer, save_method = "merged_16bit")

print("Model and adapters saved successfully.")

Model and adapters saved successfully.


## 7. GGUF Export for Local Inference
To use this model in apps like LM Studio or Ollama, we can export it to GGUF format. Unsloth supports efficient quantization during export.

In [14]:
# Export to GGUF format (e.g., q8_0 for high quality local use)
# This will save the file to your /content/ folder
model.save_pretrained_gguf(
    "meetme_gguf_model",
    tokenizer,
    quantization_method = "q8_0",
)

print("GGUF model exported successfully.")

Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/757 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in meetme_gguf_model/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


...

model-00002-of-00002.safetensors: downloading bytes:           |  0.00B            



Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [02:27<00:00, 73.91s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)




Unsloth: Merging weights into 16bit:   0%|          | 0/2 [00:00<?, ?it/s]

Unsloth: Merging weights into 16bit:  50%|█████     | 1/2 [01:15<01:15, 75.68s/it]

Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [01:55<00:00, 58.00s/it]


Unsloth: Merge process complete. Saved to `/content/meetme_gguf_model`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF q8_0 might take 3 minutes.
\        /    [2] Single-pass export: converting straight to ['q8_0'] - no separate quantize step.
 "-____-"     In total, you will have to wait at least 6 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Installing prebuilt llama.cpp b10194-mix-f08678f (app-b10194-mix-f08678f-linux-x64-cpu.tar.gz) - skipping compilation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into q8_0 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['meetme_gguf_model_gguf/qwen2.5-3b-instruct.Q8_0.gguf']
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['meetme_gguf_model_

## 7.5 Push Model to Hugging Face
Use this section to upload your fine-tuned adapters directly to your Hugging Face account.

In [53]:
from huggingface_hub import login, upload_folder

# 1. Login - This will prompt you for your Hugging Face Write Token
print("Please provide a Write token from https://huggingface.co/settings/tokens")
login()

# 2. Upload the folder containing your LoRA adapters
# We upload the 'meetme_lora_model' folder created in step 6
try:
    upload_folder(
        folder_path="meetme_lora_model",
        repo_id="EquilStable/MEETME",
        repo_type="model",
        commit_message="Initial upload of MeetMe personality twin adapters"
    )
    print("\nSuccessfully uploaded model to https://huggingface.co/EquilStable/MEETME")
except Exception as e:
    print(f"\nAn error occurred during upload: {e}")

Please provide a Write token from https://huggingface.co/settings/tokens



Successfully uploaded model to https://huggingface.co/EquilStable/MEETME


## 8. Play with the Model
Use the cell below to interact with your fine-tuned model. It includes formatting to make the conversation easy to read.

In [40]:
from IPython.display import Markdown, display, HTML

def chat_with_model(user_query):
    messages = [
        {"role": "user", "content": user_query},
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize = True,
        add_generation_prompt = True,
        return_tensors = "pt",
    ).to("cuda")

    outputs = model.generate(
        input_ids = inputs,
        attention_mask = (inputs != tokenizer.pad_token_id).long(),
        max_new_tokens = 1024,
        temperature = 0.7,
        do_sample = True,
        pad_token_id = tokenizer.pad_token_id
    )

    response = tokenizer.decode(outputs[0][len(inputs[0]):], skip_special_tokens=True)

    # Updated Visuals: Solid black background with black text on bubbles
    display(HTML(f"""
    <div style='padding: 25px; border-radius: 15px; background-color: #000000; font-family: sans-serif;'>
        <div style='background-color: #e0e0e0; padding: 12px; border-radius: 10px; margin-bottom: 12px; border-left: 6px solid #888;'>
            <div style='font-weight: bold; font-size: 0.75em; margin-bottom: 5px; color: #444; text-transform: uppercase;'>Question</div>
            <div style='color: #000000; font-size: 1.05em; font-weight: 500;'>{user_query}</div>
        </div>
        <div style='background-color: #ffffff; padding: 12px; border-radius: 10px; border-left: 6px solid #007bff;'>
            <div style='font-weight: bold; font-size: 0.75em; margin-bottom: 5px; color: #007bff; text-transform: uppercase;'>Answer</div>
            <div style='color: #000000; font-size: 1.05em; font-weight: 500;'>{response}</div>
        </div>
    </div>
    """))

# Quick test of the new look
chat_with_model("What motivates you to keep creating?")

Both `max_new_tokens` (=1024) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


## 9. Interactive Sandbox
Use the input box below to chat with your model. Type a message and press Enter to see the response.

In [65]:
import ipywidgets as widgets
from IPython.display import clear_output

input_box = widgets.Text(
    placeholder='Ask your personality twin something...',
    description='Message:',
    layout=widgets.Layout(width='80%')
)

def on_submit(sender):
    query = input_box.value
    if query.strip():
        # clear_output(wait=True) # Uncomment if you want to keep only the latest message
        chat_with_model(query)
        input_box.value = ''

input_box.on_submit(on_submit)
display(input_box)

Text(value='', description='Message:', layout=Layout(width='80%'), placeholder='Ask your personality twin some…

Both `max_new_tokens` (=1024) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


## 10. Model Comparison: Base vs. Fine-Tuned
Run the cell below to load the original un-tuned model and compare its responses side-by-side with your new personality twin.

In [64]:
from unsloth import FastLanguageModel
import torch

# Load the base model (un-tuned) for comparison
# We use 4-bit again to stay within memory limits
base_model, _ = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen2.5-3B-Instruct",
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(base_model)

def compare_models(query):
    messages = [{"role": "user", "content": query}]
    inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")

    # Generate from Fine-Tuned Model (the 'model' variable from previous cells)
    ft_outputs = model.generate(input_ids=inputs, max_new_tokens=256, temperature=0.7, do_sample=True, pad_token_id=tokenizer.pad_token_id)
    ft_response = tokenizer.decode(ft_outputs[0][len(inputs[0]):], skip_special_tokens=True)

    # Generate from Base Model
    base_outputs = base_model.generate(input_ids=inputs, max_new_tokens=256, temperature=0.7, do_sample=True, pad_token_id=tokenizer.pad_token_id)
    base_response = tokenizer.decode(base_outputs[0][len(inputs[0]):], skip_special_tokens=True)

    # Display Side-by-Side
    display(HTML(f"""
    <div style='background-color: #000; padding: 20px; border-radius: 15px; font-family: sans-serif;'>
        <div style='background-color: #e0e0e0; color: #000; padding: 10px; border-radius: 8px; margin-bottom: 15px; text-align: center;'>
            <strong>PROMPT:</strong> {query}
        </div>
        <div style='display: flex; gap: 15px;'>
            <div style='flex: 1; background-color: #fff; color: #000; padding: 12px; border-radius: 10px; border-top: 5px solid #888;'>
                <div style='font-weight: bold; font-size: 0.75em; color: #888; margin-bottom: 5px;'>BASE MODEL (Original)</div>
                {base_response}
            </div>
            <div style='flex: 1; background-color: #fff; color: #000; padding: 12px; border-radius: 10px; border-top: 5px solid #007bff;'>
                <div style='font-weight: bold; font-size: 0.75em; color: #007bff; margin-bottom: 5px;'>MEETME AI (Fine-Tuned)</div>
                {ft_response}
            </div>
        </div>
    </div>
    """))

# Try a comparison!
compare_models("How are you?")

==((====))==  Unsloth 2026.7.6: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
